<a href="https://colab.research.google.com/github/fvangool/Deep-Learning-Specialization-Coursera/blob/main/Catboostv19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install catboost wandb -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 28.3 MB/s eta 0:00:00


In [ ]:
"""
CatBoost v23 — Based on CV=0.97799 / LB=0.97726 notebook
==========================================================
SOURCE NOTEBOOK
───────────────
  Credit: ps6e4-catboost-cv-0-97799-lb-0-97726.ipynb
  CV=0.97799, LB=0.97726 — solid single-model baseline

WHAT WAS IN THE NOTEBOOK (kept)
────────────────────────────────
  + TE_ORIG prior mean encoding from original dataset
  + Domain features (ET_proxy, moist_rain, heat_stress, etc.)
  + Formula threshold features (soil_lt_25, temp_gt_30, rain_lt_300, wind_gt_10)
  + Digit extraction features (k=-4..3)
  + Categorical interaction features (all CATS pairwise)
  + Inner-CV TargetEncoder (leak-free, std/var/median stats)
  + sklearn TargetEncoder per-class (multiclass)
  + Balanced class weights
  + StratifiedKFold (10 folds)
  + CatBoostClassifier with GPU

WHAT WE ADD (not in notebook)
──────────────────────────────
  + Multi-seed averaging (5 seeds x 10 folds = 50 folds total)
  + magic_score formula feature + magic_score_soft boundary distance
  + Logit-space bias tuning on averaged OOF (proven +0.0006 BA)
  + Logit columns (logit_Low, logit_Medium, logit_High from TE_ORIG)
  + Per-class OOF BA breakdown (Low/Medium/High separately)
  + Hard-example diagnostics (pre/post bias)
  + magic_score=0 Medium recall diagnostic
  + W&B logging (fold-level + seed-level + summary)
  + Telegram notifier + heartbeat
  + Detailed in-fold console output (iter, time, per-class BA)
  + Aggressive memory cleanup between folds
  + OOF + pred .npy save for ensemble integration
  + Submission CSV with proper Low=0/Medium=1/High=2 mapping

SPEED / GPU OPTIMISATIONS
──────────────────────────
  + task_type="GPU" throughout
  + allow_writing_files=False (avoids disk I/O overhead)
  + early_stopping_rounds=150 with TotalF1 eval metric
  + Pool objects freed immediately after fit
  + float64->float32 cast before training
  + gc.collect() after each fold
  + Params hardcoded — no Optuna overhead at runtime

LABEL MAPPING
─────────────
  Low=0  Medium=1  High=2  (matches XGB/LGBM ensemble convention)

SAVE CONVENTION
───────────────
  oof_cat_v23.npy         (n_competition, 3)
  pred_cat_v23.npy        (n_test, 3)
  oof_cat_v23_raw.npy     (pre-bias)
  submission_cat_v23.csv
"""

# ============================================================
# IMPORTS
# ============================================================
import gc
import os
import json
import time
import warnings
import threading
import traceback
import urllib.request
from contextlib import contextmanager
from itertools import combinations

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wandb

from google.colab import userdata
from catboost import CatBoostClassifier, Pool

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.utils.class_weight import compute_class_weight

from scipy.special import logit

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

# ============================================================
# SECTION 0 — CONFIGURATION
# ============================================================
TRAIN_PATH = "/content/drive/MyDrive/irrigation_need_v15/train.csv"
TEST_PATH  = "/content/drive/MyDrive/irrigation_need_v15/test.csv"
ORIG_PATH  = "/content/drive/MyDrive/irrigation_need_v15/irrigation_prediction.csv"
OUT_DIR    = "/content/drive/MyDrive/irrigation_need_v15/"
SUB_DIR    = "/content/"
HARD_IDX_PATH = f"{OUT_DIR}hard_example_indices.npy"

TELEGRAM_BOT_TOKEN = "8755783601:AAGuCUzM6CdgjA825tep5f5zj0FNd7iSkp4"
TELEGRAM_CHAT_ID   = "5422067007"
TELEGRAM_ENABLED   = True

WANDB_PROJECT = "ps-s6e4-irrigation"
WANDB_ENTITY  = "wblackstone-twilight-signals"
WANDB_ENABLED = True
WB_TOKEN      = "WB_TOKEN"

TAG      = "cat_v23"
RUN_NAME = "CatBoost v23"

SEEDS     = [42, 123, 2024, 7, 314]
N_FOLDS   = 10
N_CLASSES = 3
TARGET    = "Irrigation_Need"

TARGET_MAP     = {"Low": 0, "Medium": 1, "High": 2}
INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}
CLASS_NAMES    = ["Low", "Medium", "High"]

NUMS = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm",
]
CATS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
]

# Inner-CV for TargetEncoder leak prevention
INNER_FOLDS = 5
TE_STATS    = ["std", "var", "median"]

# Magic score thresholds (formula features)
SOIL_THRESH = 25
RAIN_THRESH = 300
TEMP_THRESH = 30
WIND_THRESH = 10

# CatBoost params — GPU optimised, no Optuna overhead
CAT_PARAMS = {
    "loss_function"         : "MultiClass",
    "eval_metric"           : "TotalF1",
    "iterations"            : 3000,
    "learning_rate"         : 0.05,
    "depth"                 : 6,
    "l2_leaf_reg"           : 3.0,
    "random_strength"       : 1.0,
    "bagging_temperature"   : 0.5,
    "grow_policy"           : "SymmetricTree",
    "task_type"             : "GPU",
    "devices"               : "0",
    "early_stopping_rounds" : 150,
    "allow_writing_files"   : False,
    "verbose"               : 0,
}

print(f"{'='*60}")
print(f" {RUN_NAME}")
print(f"{'='*60}")
print(f" Seeds  : {SEEDS}")
print(f" Folds  : {N_FOLDS}")
print(f" Total  : {len(SEEDS) * N_FOLDS} folds")
print(f"{'='*60}")

# ============================================================
# SECTION 1 — W&B HELPER
# ============================================================
class WandbLogger:
    def __init__(self, enabled=True):
        self.enabled = enabled
        self.run     = None

    def init(self, config):
        if not self.enabled:
            return
        try:
            api_key = userdata.get(WB_TOKEN)
            wandb.login(key=api_key, relogin=True)
            self.run = wandb.init(
                project=WANDB_PROJECT, entity=WANDB_ENTITY,
                name=TAG, config=config,
                tags=["catboost", "v23", "ps-s6e4"],
            )
            print(f" W&B run: {self.run.url}")
        except Exception as e:
            print(f" [W&B] init failed: {e}")
            self.enabled = False

    def log(self, metrics, step=None):
        if not self.enabled or self.run is None:
            return
        try:
            wandb.log(metrics, step=step) if step is not None \
                else wandb.log(metrics)
        except Exception as e:
            print(f" [W&B] log failed: {e}")

    def log_confusion_matrix(self, y_true, y_pred, title):
        if not self.enabled or self.run is None:
            return
        try:
            wandb.log({title: wandb.plot.confusion_matrix(
                probs=None, y_true=y_true.tolist(),
                preds=y_pred.tolist(), class_names=CLASS_NAMES,
            )})
        except Exception as e:
            print(f" [W&B] confusion matrix failed: {e}")

    def log_artifact(self, local_path, artifact_name,
                     artifact_type, description=""):
        if not self.enabled or self.run is None:
            return
        try:
            art = wandb.Artifact(name=artifact_name, type=artifact_type,
                                 description=description)
            art.add_file(local_path)
            self.run.log_artifact(art)
        except Exception as e:
            print(f" [W&B] artifact failed: {e}")

    def summary(self, metrics):
        if not self.enabled or self.run is None:
            return
        try:
            for k, v in metrics.items():
                wandb.run.summary[k] = v
        except Exception as e:
            print(f" [W&B] summary failed: {e}")

    def finish(self):
        if not self.enabled or self.run is None:
            return
        try:
            wandb.finish()
        except Exception as e:
            print(f" [W&B] finish failed: {e}")


wb = WandbLogger(enabled=WANDB_ENABLED)

# ============================================================
# SECTION 2 — TELEGRAM NOTIFIER
# ============================================================
class TelegramNotifier:
    def __init__(self, bot_token=TELEGRAM_BOT_TOKEN,
                 chat_id=TELEGRAM_CHAT_ID,
                 enabled=TELEGRAM_ENABLED, run_name=RUN_NAME):
        self.bot_token  = bot_token
        self.chat_id    = chat_id
        self.enabled    = enabled
        self.run_name   = run_name
        self._start     = None
        self._hb_stop   = threading.Event()
        self._hb_thread = None

    def send(self, message, silent=False):
        if not self.enabled:
            return True
        try:
            url = f"https://api.telegram.org/bot{self.bot_token}/sendMessage"
            payload = json.dumps({
                "chat_id": self.chat_id, "text": message,
                "disable_notification": silent,
            }).encode("utf-8")
            req = urllib.request.Request(
                url, data=payload,
                headers={"Content-Type": "application/json"},
            )
            urllib.request.urlopen(req, timeout=10)
            return True
        except Exception as e:
            print(f"[TELEGRAM] {e}")
            return False

    def start_timer(self):
        self._start = time.time()
        return self

    def elapsed(self):
        if self._start is None:
            return "unknown"
        s = int(time.time() - self._start)
        h, r = divmod(s, 3600)
        m, s = divmod(r, 60)
        return f"{h}h {m}m {s}s" if h else f"{m}m {s}s"

    def heartbeat(self, interval_minutes=20):
        if not self.enabled:
            return self
        if self._hb_thread is not None:
            self._hb_stop.set()
        self._hb_stop = threading.Event()
        def _loop():
            count = 0
            while not self._hb_stop.wait(interval_minutes * 60):
                count += 1
                self.send(f"[{self.run_name}] running | "
                          f"{self.elapsed()} | hb#{count}", silent=True)
        self._hb_thread = threading.Thread(target=_loop, daemon=True)
        self._hb_thread.start()
        return self

    def stop_heartbeat(self):
        if self._hb_stop:
            self._hb_stop.set()

    def notify_seed(self, seed_idx, n_seeds, seed, seed_ba, silent=True):
        bar = "X" * seed_idx + "." * (n_seeds - seed_idx)
        self.send(
            f"[{self.run_name}] Seed {seed_idx}/{n_seeds} [{bar}]\n"
            f" Seed={seed} | OOF BA={seed_ba:.6f} | {self.elapsed()}",
            silent=silent,
        )

    def success(self, oof_score, extra=""):
        self.stop_heartbeat()
        msg = (f"[{self.run_name}] Complete\n"
               f" OOF BA={oof_score:.6f} | Runtime={self.elapsed()}")
        if extra:
            msg += f"\n {extra}"
        self.send(msg)

    def failure(self, exc=None, context=""):
        self.stop_heartbeat()
        tb = traceback.format_exc() if exc else ""
        if len(tb) > 800:
            tb = "..." + tb[-800:]
        msg = f"[{self.run_name}] FAILED | {self.elapsed()}"
        if context:
            msg += f"\n {context}"
        if exc:
            msg += f"\n {type(exc).__name__}: {exc}"
        if tb:
            msg += f"\n{tb}"
        self.send(msg)

    @contextmanager
    def run_context(self, context=""):
        try:
            yield
        except Exception as e:
            self.failure(exc=e, context=context)
            raise


notifier = TelegramNotifier()

# ============================================================
# SECTION 3 — BIAS TUNING
# ============================================================
def tune_logit_bias(oof_probs, y_true):
    def get_preds(probs, bias):
        adj = logit(np.clip(probs, 1e-15, 1 - 1e-15)) + bias
        return np.argmax(adj, axis=1)

    best_bias   = np.zeros(3)
    best_score  = balanced_accuracy_score(y_true, oof_probs.argmax(1))
    raw_score   = best_score
    opt_history = [best_score]

    for step in [1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002]:
        improved = True
        while improved:
            improved = False
            for ci in range(3):
                for d in [1, -1]:
                    trial      = best_bias.copy()
                    trial[ci] += d * step
                    s = balanced_accuracy_score(
                        y_true, get_preds(oof_probs, trial)
                    )
                    if s > best_score + 1e-9:
                        best_score, best_bias, improved = s, trial, True
                        opt_history.append(best_score)

    print(f"  Bias raw  : {raw_score:.6f}")
    print(f"  Bias tuned: {best_score:.6f} (+{best_score - raw_score:.6f})")
    print(f"  Biases    : Low={best_bias[0]:.4f} "
          f"Med={best_bias[1]:.4f} High={best_bias[2]:.4f}")
    return best_bias, best_score, opt_history


def apply_bias(probs, bias):
    log_p = logit(np.clip(probs, 1e-15, 1 - 1e-15)) + bias
    exp_p = np.exp(log_p)
    return (exp_p / exp_p.sum(axis=1, keepdims=True)).astype(np.float32)

# ============================================================
# SECTION 4 — FEATURE ENGINEERING
# ============================================================
def add_te_orig(train, test, orig):
    """
    Leak-free target encoding from original dataset.
    Maps each CATS+NUMS column to mean target in orig.
    Also adds logit columns for Low/Med/High probabilities.
    """
    created = []
    for col in CATS + NUMS:
        if orig[col].nunique() == 0:
            continue
        name    = f"TE_ORIG_{col}"
        mapping = orig.groupby(col)["target"].mean().astype("float32")
        for df in [train, test, orig]:
            df[name] = df[col].astype(object).map(mapping).fillna(0.5).astype("float32")
        created.append(name)

    # Logit features from per-class TE_ORIG
    for col in CATS + NUMS:
        for cls in range(3):
            name    = f"TE_ORIG_{col}_cls{cls}"
            mapping = (orig.groupby(col)["target"]
                       .apply(lambda x: (x == cls).mean())
                       .astype("float32"))
            for df in [train, test, orig]:
                p = df[col].astype(object).map(mapping).fillna(1/3).clip(1e-6, 1-1e-6)
                df[f"logit_TE_{col}_cls{cls}"] = logit(p).astype("float32")
            created.append(f"logit_TE_{col}_cls{cls}")

    return created


def add_domain_features(df):
    """Domain knowledge features from the notebook — kept exactly."""
    df["moist_rain"]    = df["Soil_Moisture"] / (df["Rainfall_mm"] + 1)
    df["moist_temp"]    = df["Soil_Moisture"] / (df["Temperature_C"] + 1)
    df["moist_wind"]    = df["Soil_Moisture"] / (df["Wind_Speed_kmh"] + 1)
    df["ET_proxy"]      = (df["Temperature_C"] * df["Wind_Speed_kmh"]
                           * df["Sunlight_Hours"]) / (df["Humidity"] + 1)
    df["heat_stress"]   = df["Temperature_C"] * df["Sunlight_Hours"]
    df["drying_force"]  = (df["Wind_Speed_kmh"] * df["Temperature_C"]
                           / (df["Humidity"] + 1))
    df["water_supply"]  = df["Rainfall_mm"] + df["Previous_Irrigation_mm"]
    df["water_deficit"] = df["Soil_Moisture"] - df["water_supply"] * 0.1
    df["soil_quality"]  = df["Organic_Carbon"] / (df["Electrical_Conductivity"] + 0.1)
    df["moist_x_wind"]  = df["Soil_Moisture"] * df["Wind_Speed_kmh"]
    df["moist_x_temp"]  = df["Soil_Moisture"] * df["Temperature_C"]
    df["wind_x_temp"]   = df["Wind_Speed_kmh"] * df["Temperature_C"]
    df["moisture_sq"]   = df["Soil_Moisture"] ** 2
    df["wind_sq"]       = df["Wind_Speed_kmh"] ** 2
    df["temp_sq"]       = df["Temperature_C"] ** 2
    return [
        "moist_rain", "moist_temp", "moist_wind", "ET_proxy", "heat_stress",
        "drying_force", "water_supply", "water_deficit", "soil_quality",
        "moist_x_wind", "moist_x_temp", "wind_x_temp",
        "moisture_sq", "wind_sq", "temp_sq",
    ]


def add_formula_features(df):
    """
    Notebook threshold features + magic_score formula (our addition).
    magic_score = 2*soil + 2*rain + temp + wind - 2*harvest - 2*sowing - mulch
    """
    # Notebook originals
    df["soil_lt_25"]  = (df["Soil_Moisture"]     < SOIL_THRESH).astype("int32")
    df["temp_gt_30"]  = (df["Temperature_C"]     > TEMP_THRESH).astype("int32")
    df["rain_lt_300"] = (df["Rainfall_mm"]       < RAIN_THRESH).astype("int32")
    df["wind_gt_10"]  = (df["Wind_Speed_kmh"]    > WIND_THRESH).astype("int32")
    df["is_harvest"]  = (df["Crop_Growth_Stage"] == "Harvest").astype("int32")
    df["is_sowing"]   = (df["Crop_Growth_Stage"] == "Sowing").astype("int32")
    df["mulching_yes"]= (df["Mulching_Used"]     == "Yes").astype("int32")

    # Magic score (our addition — not in notebook)
    df["magic_score"] = (
        2 * df["soil_lt_25"]
        + 2 * df["rain_lt_300"]
        + df["temp_gt_30"]
        + df["wind_gt_10"]
        - 2 * df["is_harvest"]
        - 2 * df["is_sowing"]
        - df["mulching_yes"]
    )
    # Soft boundary distance from magic_score=0
    df["magic_dist_0"] = df["magic_score"].abs().astype("float32")
    df["at_boundary"]  = (df["magic_score"] == 0).astype("int32")

    return [
        "soil_lt_25", "temp_gt_30", "rain_lt_300", "wind_gt_10",
        "is_harvest", "is_sowing", "mulching_yes",
        "magic_score", "magic_dist_0", "at_boundary",
    ]


def add_digit_features(df, num_cols, k_min=-4, k_max=3):
    """Digit extraction — from notebook, kept exactly."""
    created = []
    for c in num_cols:
        for k in range(k_min, k_max + 1):
            name = f"{c}_digit{k}"
            df[name] = ((df[c] * (10 ** k)) % 10).astype("int8")
            created.append(name)
    # Drop zero-variance
    drop = [c for c in created if df[c].nunique() == 1]
    if drop:
        df.drop(columns=drop, inplace=True)
        created = [c for c in created if c not in drop]
    print(f"  Digit features: {len(created)} kept, {len(drop)} dropped (zero-variance)")
    return created


def add_cat_interactions(dfs):
    """All pairwise categorical interactions — from notebook."""
    created = []
    for i, c1 in enumerate(CATS[:-1]):
        for c2 in CATS[i+1:]:
            name = f"{c1}_{c2}"
            for df in dfs:
                df[name] = df[c1].astype(str) + "_" + df[c2].astype(str)
            created.append(name)
    print(f"  Cat interaction features: {len(created)}")
    return created

# ============================================================
# SECTION 5 — MAIN PIPELINE
# ============================================================
if __name__ == "__main__":
    notifier.start_timer().heartbeat(interval_minutes=20)

    wb.init(config={
        "seeds"          : SEEDS,
        "n_folds"        : N_FOLDS,
        "inner_folds"    : INNER_FOLDS,
        "total_cv_folds" : len(SEEDS) * N_FOLDS,
        "tag"            : TAG,
        "version"        : "v23",
        **{f"cat_{k}": v for k, v in CAT_PARAMS.items()},
    })

    notifier.send(
        f"[{RUN_NAME}] Starting\n"
        f" Seeds: {SEEDS}\n"
        f" {len(SEEDS)} x {N_FOLDS}-fold = {len(SEEDS)*N_FOLDS} folds",
        silent=True,
    )

    try:
        # ── 1. Load data ──────────────────────────────────────
        print(f"\n[1] Loading data...")
        train_raw = pd.read_csv(TRAIN_PATH)
        test_raw  = pd.read_csv(TEST_PATH)
        orig_raw  = pd.read_csv(ORIG_PATH)

        if "Irrigation_Requirement" in orig_raw.columns:
            orig_raw = orig_raw.rename(
                columns={"Irrigation_Requirement": TARGET}
            )

        train_raw["target"] = train_raw[TARGET].map(TARGET_MAP)
        orig_raw["target"]  = orig_raw[TARGET].map(TARGET_MAP)

        n_competition = len(train_raw)
        test_ids      = test_raw["id"].copy()

        # Hard examples
        hard_mask = None
        if os.path.exists(HARD_IDX_PATH):
            hard_idx  = np.load(HARD_IDX_PATH)
            hard_mask = np.zeros(n_competition, dtype=bool)
            hard_mask[hard_idx] = True
            print(f"  Hard examples: {hard_mask.sum():,} rows")

        # Working copies
        train = train_raw.drop(columns=[TARGET]).copy()
        test  = test_raw.drop(columns=["id"]).copy()
        orig  = orig_raw.drop(columns=[TARGET]).copy()

        print(f"  Competition: {n_competition:,} | "
              f"Test: {len(test):,} | Orig: {len(orig):,}")

        # ── 2. Feature engineering ────────────────────────────
        print(f"\n[2] Feature engineering...")

        # TE_ORIG + logit columns (our addition)
        print("  TE_ORIG prior encoding...")
        te_orig_cols = add_te_orig(train, test, orig)
        print(f"  TE_ORIG: {len(te_orig_cols)} features")

        # Domain features (notebook)
        print("  Domain features...")
        domain_cols  = add_domain_features(train)
        _            = add_domain_features(test)
        _            = add_domain_features(orig)

        # Formula + magic_score (notebook + our addition)
        print("  Formula + magic_score features...")
        formula_cols = add_formula_features(train)
        _            = add_formula_features(test)
        _            = add_formula_features(orig)

        # Digit features (notebook)
        print("  Digit features...")
        train_digits = add_digit_features(train, NUMS)
        test_digits  = add_digit_features(test,  NUMS)
        # Align test digits to train (some may differ due to nunique)
        for c in train_digits:
            if c not in test.columns:
                test[c] = 0
        test_digits = train_digits

        # Cat interactions (notebook)
        print("  Categorical interactions...")
        cat_inter_cols = add_cat_interactions([train, test, orig])

        # Numerical category columns (notebook)
        print("  Numerical category columns...")
        num_cat_cols = []
        for col in NUMS:
            nc = f"{col}_cat"
            train[nc] = train[col].astype(str).astype("category")
            test[nc]  = test[col].astype(str).astype("category")
            orig[nc]  = orig[col].astype(str).astype("category")
            num_cat_cols.append(nc)
        print(f"  Num-cat features: {len(num_cat_cols)}")

        # Categorical columns for CatBoost
        CATS_FE = CATS + cat_inter_cols + num_cat_cols
        for c in CATS_FE:
            train[c] = train[c].astype(str)
            test[c]  = test[c].astype(str)
            if c in orig.columns:
                orig[c] = orig[c].astype(str)

        # Float32 cast
        float_cols = [c for c in train.columns
                      if train[c].dtype == "float64"]
        train[float_cols] = train[float_cols].astype("float32")
        test[float_cols]  = test[float_cols].astype("float32")

        # Feature list (all except id, target, TARGET)
        drop_cols  = {"id", "target", TARGET}
        FINAL_FEATS = [c for c in train.columns if c not in drop_cols]
        # TE columns for inner-CV encoding
        TE_COLUMNS = num_cat_cols

        y_train = train["target"].values.astype(int)
        y_orig  = orig["target"].values.astype(int)

        # Class weights
        classes = np.unique(y_train)
        cw      = compute_class_weight(
            class_weight="balanced", classes=classes, y=y_train
        )
        weights_dict = {int(k): float(v) for k, v in zip(classes, cw)}

        print(f"\n  Final features: {len(FINAL_FEATS)}")
        print(f"  Cat features  : {len(CATS_FE)}")
        print(f"  Class weights : {weights_dict}")
        wb.log({
            "n_features" : len(FINAL_FEATS),
            "n_cat_feats": len(CATS_FE),
        })

        # ── 3. Multi-seed CV ──────────────────────────────────
        print(f"\n[3] Multi-seed CV "
              f"({len(SEEDS)} seeds x {N_FOLDS} folds)...")

        oof_accum       = np.zeros(
            (n_competition, N_CLASSES), dtype=np.float64
        )
        test_accum      = np.zeros(
            (len(test), N_CLASSES), dtype=np.float64
        )
        seed_oof_scores = []
        all_best_iters  = []
        n_folds_total   = len(SEEDS) * N_FOLDS
        global_fold_num = 0

        for seed_idx, seed in enumerate(SEEDS):
            seed_start = time.time()
            print(f"\n{'='*60}")
            print(f" SEED {seed} ({seed_idx+1}/{len(SEEDS)})")
            print(f"{'='*60}")

            kf       = StratifiedKFold(
                n_splits=N_FOLDS, shuffle=True, random_state=seed
            )
            in_skf   = StratifiedKFold(
                n_splits=INNER_FOLDS, shuffle=True, random_state=seed
            )
            oof_seed = np.zeros(
                (n_competition, N_CLASSES), dtype=np.float64
            )

            X = train[FINAL_FEATS].copy()
            y = pd.Series(y_train)

            for fold, (tr_idx, val_idx) in enumerate(
                kf.split(X, y), 1
            ):
                fold_start = time.time()
                print(f"\n  Fold {fold}/{N_FOLDS} | Seed {seed}")

                with notifier.run_context(f"Seed {seed} Fold {fold}"):
                    X_tr  = X.iloc[tr_idx].copy()
                    X_val = X.iloc[val_idx].copy()
                    y_tr  = y.iloc[tr_idx].copy()
                    y_val = y.iloc[val_idx].copy()
                    X_tst = test[FINAL_FEATS].copy()

                    X_tr["target"] = y_tr.values

                    # ── Inner-CV TargetEncoder (std/var/median) ──
                    # Leak-free: computed on inner folds, applied
                    # to outer val and test — exactly as in notebook
                    te_stat_cols = [
                        f"TE1_{col}_{s}"
                        for col in TE_COLUMNS
                        for s in TE_STATS
                    ]
                    for c in te_stat_cols:
                        X_tr[c]  = 0.0
                        X_tst[c] = 0.0

                    for _, (in_tr, in_va) in enumerate(
                        in_skf.split(X_tr, y_tr)
                    ):
                        X_tr2 = X_tr.iloc[in_tr].copy()
                        for col in TE_COLUMNS:
                            tmp = X_tr2.groupby(
                                col, observed=False
                            )["target"].agg(TE_STATS)
                            tmp.columns = [
                                f"TE1_{col}_{s}" for s in TE_STATS
                            ]
                            for c in tmp.columns:
                                vals = (
                                    X_tr.iloc[in_va][col]
                                    .map(tmp[c])
                                )
                                vals = (
                                    pd.to_numeric(vals, errors="coerce")
                                    .fillna(0).astype("float32").values
                                )
                                X_tr.iloc[
                                    in_va,
                                    X_tr.columns.get_loc(c)
                                ] = vals

                    # Apply from X_tr to X_val and X_tst
                    for col in TE_COLUMNS:
                        tmp = X_tr.groupby(
                            col, observed=False
                        )["target"].agg(TE_STATS)
                        tmp.columns = [
                            f"TE1_{col}_{s}" for s in TE_STATS
                        ]
                        for c in tmp.columns:
                            X_val[c] = (
                                pd.to_numeric(
                                    X_val[col].map(tmp[c]),
                                    errors="coerce",
                                ).fillna(0).astype("float32").values
                            )
                            X_tr[c] = (
                                pd.to_numeric(
                                    X_tr[c], errors="coerce"
                                ).fillna(0).astype("float32").values
                            )
                            X_tst[c] = (
                                pd.to_numeric(
                                    X_tst[col].map(tmp[c]),
                                    errors="coerce",
                                ).fillna(0).astype("float32").values
                            )

                    # ── sklearn multiclass TargetEncoder ────────
                    te_all_cols  = TE_COLUMNS + CATS
                    TE = TargetEncoder(
                        cv=INNER_FOLDS,
                        random_state=seed,
                        target_type="multiclass",
                        shuffle=True,
                    )
                    enc_tr = TE.fit_transform(X_tr[te_all_cols], y_tr)
                    n_cls  = len(TE.classes_)
                    sk_te_cols = [
                        f"TE_{col}_cls{c}"
                        for col in te_all_cols
                        for c in TE.classes_
                    ]
                    X_tr[sk_te_cols]  = enc_tr
                    X_val[sk_te_cols] = TE.transform(X_val[te_all_cols])
                    X_tst[sk_te_cols] = TE.transform(X_tst[te_all_cols])

                    # Drop TE_COLUMNS (num_cat raw) + target
                    for df in [X_tr, X_val, X_tst]:
                        df.drop(
                            columns=[c for c in TE_COLUMNS
                                     if c in df.columns],
                            inplace=True, errors="ignore",
                        )
                    X_tr.drop(columns=["target"],
                               inplace=True, errors="ignore")

                    # Final cat feature list for this fold
                    cats_in_fold = [
                        c for c in CATS_FE if c in X_tr.columns
                    ]
                    for df in [X_tr, X_val, X_tst]:
                        for c in cats_in_fold:
                            df[c] = df[c].astype(str)

                    # Sample weights
                    sw = y_tr.map(weights_dict).values

                    # ── CatBoost training ────────────────────────
                    params = {**CAT_PARAMS, "random_seed": seed}
                    model  = CatBoostClassifier(**params)

                    train_pool = Pool(
                        X_tr, y_tr.values,
                        cat_features=cats_in_fold,
                        weight=sw,
                    )
                    val_pool   = Pool(
                        X_val, y_val.values,
                        cat_features=cats_in_fold,
                    )
                    model.fit(train_pool, eval_set=val_pool)

                    best_iter = model.best_iteration_
                    all_best_iters.append(best_iter)

                    val_proba  = model.predict_proba(X_val)
                    test_proba = model.predict_proba(X_tst)

                    oof_seed[val_idx] += val_proba
                    test_accum        += test_proba / n_folds_total

                    del train_pool, val_pool
                    gc.collect()

                    # ── Detailed fold diagnostics ─────────────────
                    fold_ba = balanced_accuracy_score(
                        y_val, val_proba.argmax(1)
                    )
                    elapsed = int(time.time() - fold_start)

                    # Per-class fold BA
                    per_class_str = ""
                    for cls in range(3):
                        m   = (y_val == cls).values
                        cba = (val_proba[m].argmax(1) == cls).mean() \
                              if m.sum() > 0 else 0.0
                        per_class_str += (
                            f" {CLASS_NAMES[cls]}={cba:.4f}"
                        )

                    print(f"    BA={fold_ba:.5f} | "
                          f"iter={best_iter} | "
                          f"{elapsed//60}m {elapsed%60}s")
                    print(f"    Per-class:{per_class_str}")

                    global_fold_num += 1
                    wb.log({
                        "fold_ba"       : fold_ba,
                        "fold_best_iter": best_iter,
                        "fold_elapsed_s": elapsed,
                        "seed"          : seed,
                        "fold"          : fold,
                        "seed_idx"      : seed_idx + 1,
                        "global_fold"   : global_fold_num,
                    }, step=global_fold_num)

                    del model, X_tr, X_val, X_tst
                    gc.collect()

            # Seed-level OOF
            seed_ba = balanced_accuracy_score(
                y_train, oof_seed.argmax(1)
            )
            seed_oof_scores.append(seed_ba)
            oof_accum += oof_seed / len(SEEDS)

            seed_elapsed = int(time.time() - seed_start)
            print(f"\n  Seed {seed} OOF BA: {seed_ba:.6f} | "
                  f"Avg iter: {int(np.mean(all_best_iters[-N_FOLDS:]))} | "
                  f"{seed_elapsed//60}m {seed_elapsed%60}s")

            wb.log({
                f"seed_{seed}_oof_ba": seed_ba,
                "seed_oof_ba"        : seed_ba,
                "seed_elapsed_s"     : seed_elapsed,
            })
            notifier.notify_seed(
                seed_idx + 1, len(SEEDS), seed, seed_ba
            )

        avg_iter = int(np.mean(all_best_iters))
        print(f"\n{'='*60}")
        print(f"Multi-seed CV complete")
        print(f"Per-seed BAs : {[round(s, 5) for s in seed_oof_scores]}")
        print(f"Mean seed BA : {np.mean(seed_oof_scores):.6f} "
              f"+/- {np.std(seed_oof_scores):.6f}")
        print(f"Avg best iter: {avg_iter}")

        # ── 4. Bias tuning ────────────────────────────────────
        print(f"\n[4] Bias tuning on averaged OOF...")
        raw_ba = balanced_accuracy_score(
            y_train, oof_accum.argmax(1)
        )
        print(f"  Averaged OOF raw BA: {raw_ba:.6f}")

        best_bias, tuned_ba, opt_history = tune_logit_bias(
            oof_accum.astype(np.float32), y_train
        )
        oof_calibrated  = apply_bias(
            oof_accum.astype(np.float32), best_bias
        )
        test_calibrated = apply_bias(
            test_accum.astype(np.float32), best_bias
        )

        per_class_ba = {}
        print(f"\n  Per-class OOF BA (post-bias):")
        for cls in range(3):
            mask = y_train == cls
            ba   = (oof_calibrated[mask].argmax(1) == cls).mean()
            per_class_ba[f"oof_ba_{CLASS_NAMES[cls].lower()}"] = float(ba)
            print(f"    {CLASS_NAMES[cls]:<8}: {ba:.5f}")

        wb.log({
            "oof_ba_raw"      : raw_ba,
            "oof_ba_biased"   : tuned_ba,
            "bias_correction" : tuned_ba - raw_ba,
            "avg_best_iter"   : avg_iter,
            "mean_seed_ba"    : float(np.mean(seed_oof_scores)),
            "std_seed_ba"     : float(np.std(seed_oof_scores)),
            "bias_low"        : float(best_bias[0]),
            "bias_medium"     : float(best_bias[1]),
            "bias_high"       : float(best_bias[2]),
            **per_class_ba,
        })
        wb.log_confusion_matrix(
            y_train, oof_calibrated.argmax(1), "oof_confusion_matrix"
        )

        # ── 5. Hard-example diagnostics ───────────────────────
        if hard_mask is not None:
            hard_oof_raw = oof_accum.astype(np.float32)[hard_mask]
            hard_oof_cal = oof_calibrated[hard_mask]
            hard_true    = y_train[hard_mask]
            hard_ba_pre  = balanced_accuracy_score(
                hard_true, hard_oof_raw.argmax(1)
            )
            hard_ba_post = balanced_accuracy_score(
                hard_true, hard_oof_cal.argmax(1)
            )
            print(f"\n  Hard-example BA pre-bias : {hard_ba_pre:.5f}")
            print(f"  Hard-example BA post-bias: {hard_ba_post:.5f}")
            print(f"  Hard-row per-class:")
            for cls in range(3):
                m = hard_true == cls
                if m.sum() == 0:
                    continue
                pre  = (hard_oof_raw[m].argmax(1) == cls).mean()
                post = (hard_oof_cal[m].argmax(1) == cls).mean()
                print(f"    {CLASS_NAMES[cls]:<8}: "
                      f"{pre:.5f} -> {post:.5f} ({post-pre:+.5f})")
            wb.log({
                "hard_oof_ba_pre_bias" : hard_ba_pre,
                "hard_oof_ba_post_bias": hard_ba_post,
            })
            wb.log_confusion_matrix(
                hard_true, hard_oof_cal.argmax(1),
                "hard_example_confusion_matrix",
            )

        # ── 6. magic_score=0 boundary analysis ───────────────
        print(f"\n[5] magic_score=0 boundary analysis...")
        s  = (train_raw["Soil_Moisture"]     < SOIL_THRESH).astype(int)
        r  = (train_raw["Rainfall_mm"]       < RAIN_THRESH).astype(int)
        t  = (train_raw["Temperature_C"]     > TEMP_THRESH).astype(int)
        w  = (train_raw["Wind_Speed_kmh"]    > WIND_THRESH).astype(int)
        h  = (train_raw["Crop_Growth_Stage"] == "Harvest").astype(int)
        sw = (train_raw["Crop_Growth_Stage"] == "Sowing").astype(int)
        m  = (train_raw["Mulching_Used"]     == "Yes").astype(int)
        ms = (2*s + 2*r + t + w - 2*h - 2*sw - m).values

        ms0_mask         = ms == 0
        y_ms0            = y_train[ms0_mask]
        p_ms0_raw        = oof_accum[ms0_mask].astype(np.float32)
        p_ms0_cal        = oof_calibrated[ms0_mask]
        med_recall_raw   = (
            p_ms0_raw[y_ms0==1].argmax(1) == 1
        ).mean() if (y_ms0==1).sum() > 0 else 0.0
        med_recall_cal   = (
            p_ms0_cal[y_ms0==1].argmax(1) == 1
        ).mean() if (y_ms0==1).sum() > 0 else 0.0
        print(f"  magic_score=0 rows : {ms0_mask.sum():,}")
        print(f"  Medium recall (raw): {med_recall_raw:.5f}")
        print(f"  Medium recall (cal): {med_recall_cal:.5f}")
        wb.log({
            "ms0_medium_recall_raw": float(med_recall_raw),
            "ms0_medium_recall_cal": float(med_recall_cal),
        })

        # ── 7. Diagnostic plots ───────────────────────────────
        print(f"\n[6] Generating diagnostic plots...")

        fig1, ax1 = plt.subplots(figsize=(9, 4))
        bars = ax1.bar(
            [str(s) for s in SEEDS], seed_oof_scores,
            color="steelblue", alpha=0.8, edgecolor="black",
        )
        ax1.axhline(
            np.mean(seed_oof_scores), color="red", linestyle="--",
            label=f"Mean: {np.mean(seed_oof_scores):.5f}",
        )
        for bar, val in zip(bars, seed_oof_scores):
            ax1.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.0001,
                f"{val:.5f}", ha="center", va="bottom", fontsize=9,
            )
        ax1.set_title(
            f"Per-Seed OOF BA ({RUN_NAME})", fontweight="bold"
        )
        ax1.set_xlabel("Seed")
        ax1.set_ylabel("Balanced Accuracy")
        ax1.legend()
        plt.tight_layout()
        wb.log({"chart_seed_oof_ba": wandb.Image(fig1)})
        plt.show()
        plt.close(fig1)

        fig2, ax2 = plt.subplots(figsize=(10, 4))
        ax2.plot(
            range(len(opt_history)), opt_history,
            color="steelblue", marker="o", markersize=4, linewidth=1.5,
        )
        ax2.axhline(
            raw_ba, color="gray", linestyle="--",
            label="Raw averaged OOF BA",
        )
        ax2.set_title(
            f"Bias Tuning: {raw_ba:.5f} -> {tuned_ba:.5f} "
            f"(+{tuned_ba - raw_ba:.5f})", fontweight="bold",
        )
        ax2.set_xlabel("Step")
        ax2.set_ylabel("Balanced Accuracy")
        ax2.legend()
        plt.tight_layout()
        wb.log({"chart_bias_tuning": wandb.Image(fig2)})
        plt.show()
        plt.close(fig2)

        fig3, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.kdeplot(
            oof_calibrated.max(1), ax=axes[0], label="OOF",
            fill=True, color="steelblue", alpha=0.5,
        )
        sns.kdeplot(
            test_calibrated.max(1), ax=axes[0], label="Test",
            fill=True, color="orange", alpha=0.3,
        )
        axes[0].set_title("Confidence Distribution")
        axes[0].legend()
        oof_dist  = pd.Series(
            [INV_TARGET_MAP[p] for p in oof_calibrated.argmax(1)]
        ).value_counts(normalize=True).sort_index()
        test_dist = pd.Series(
            [INV_TARGET_MAP[p] for p in test_calibrated.argmax(1)]
        ).value_counts(normalize=True).sort_index()
        x = np.arange(3)
        axes[1].bar(x - 0.2, oof_dist.values,  0.4,
                    label="OOF",  color="steelblue", alpha=0.7)
        axes[1].bar(x + 0.2, test_dist.values, 0.4,
                    label="Test", color="orange",    alpha=0.7)
        axes[1].set_title("Class Distribution")
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(oof_dist.index)
        axes[1].legend()
        plt.tight_layout()
        wb.log({"chart_class_dist": wandb.Image(fig3)})
        plt.show()
        plt.close(fig3)

        # ── 8. Save OOF + predictions ─────────────────────────
        print(f"\n[7] Saving...")
        oof_path  = f"{OUT_DIR}oof_{TAG}.npy"
        pred_path = f"{OUT_DIR}pred_{TAG}.npy"

        np.save(oof_path,                          oof_calibrated)
        np.save(pred_path,                         test_calibrated)
        np.save(f"{OUT_DIR}oof_{TAG}_biased.npy",  oof_calibrated)
        np.save(f"{OUT_DIR}pred_{TAG}_biased.npy", test_calibrated)
        np.save(f"{OUT_DIR}oof_{TAG}_raw.npy",
                oof_accum.astype(np.float32))

        assert oof_calibrated.shape  == (n_competition, N_CLASSES)
        assert test_calibrated.shape == (len(test_raw), N_CLASSES)
        print(f"  oof_{TAG}.npy   {oof_calibrated.shape}")
        print(f"  pred_{TAG}.npy  {test_calibrated.shape}")

        wb.log_artifact(oof_path,  f"oof_{TAG}",  "model_output",
                        f"CatBoost OOF | biased BA={tuned_ba:.6f}")
        wb.log_artifact(pred_path, f"pred_{TAG}", "model_output",
                        f"CatBoost pred | biased BA={tuned_ba:.6f}")

        # ── 9. Submission ─────────────────────────────────────
        sub = pd.DataFrame({
            "id"  : test_ids,
            TARGET: [INV_TARGET_MAP[p]
                     for p in test_calibrated.argmax(1)],
        })
        sub_path = f"{SUB_DIR}submission_{TAG}.csv"
        sub.to_csv(sub_path, index=False)
        print(f"\n  Submission: {sub_path}")
        print(sub[TARGET].value_counts().to_string())

        wb.log({
            f"submission_n_{k.lower()}": int(v)
            for k, v in sub[TARGET].value_counts().items()
        })

        # ── 10. Final summary ─────────────────────────────────
        wb.summary({
            "oof_ba_raw"     : raw_ba,
            "oof_ba_biased"  : tuned_ba,
            "bias_correction": tuned_ba - raw_ba,
            "mean_seed_ba"   : float(np.mean(seed_oof_scores)),
            "avg_best_iter"  : avg_iter,
            "tag"            : TAG,
        })

        print(f"\n{'='*60}")
        print(f" {RUN_NAME} SUMMARY")
        print(f"{'='*60}")
        print(f" Seeds        : {SEEDS}")
        print(f" Per-seed BAs : "
              f"{[round(s, 5) for s in seed_oof_scores]}")
        print(f" Mean seed BA : {np.mean(seed_oof_scores):.6f} "
              f"+/- {np.std(seed_oof_scores):.6f}")
        print(f" Raw OOF BA   : {raw_ba:.6f}")
        print(f" Biased BA    : {tuned_ba:.6f} "
              f"(+{tuned_ba - raw_ba:.6f})")
        print(f" Biases       : {np.round(best_bias, 4)}")
        print(f" Avg best iter: {avg_iter}")
        print(f" Features     : {len(FINAL_FEATS)}")
        print(f" Runtime      : {notifier.elapsed()}")
        print(f"{'='*60}")

        notifier.success(
            oof_score=tuned_ba,
            extra=(f"raw={raw_ba:.5f} | "
                   f"avg_iter={avg_iter} | "
                   f"features={len(FINAL_FEATS)}"),
        )

    except Exception as e:
        notifier.failure(exc=e, context=f"Main {RUN_NAME}")
        raise

    finally:
        wb.finish()

 CatBoost v23
 Seeds  : [42, 123, 2024, 7, 314]
 Folds  : 10
 Total  : 50 folds


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: wblackstone (wblackstone-twilight-signals) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


 W&B run: https://wandb.ai/wblackstone-twilight-signals/ps-s6e4-irrigation/runs/fg8prx62

[1] Loading data...
  Hard examples: 7,377 rows
  Competition: 630,000 | Test: 270,000 | Orig: 10,000

[2] Feature engineering...
  TE_ORIG prior encoding...
  TE_ORIG: 76 features
  Domain features...
  Formula + magic_score features...
  Digit features...
  Digit features: 54 kept, 34 dropped (zero-variance)
  Digit features: 54 kept, 34 dropped (zero-variance)
  Categorical interactions...
  Cat interaction features: 28
  Numerical category columns...
  Num-cat features: 11

  Final features: 213
  Cat features  : 47
  Class weights : {0: 0.5676949153458749, 1: 0.8783891180136694, 2: 9.995716121662145}

[3] Multi-seed CV (5 seeds x 10 folds)...

 SEED 42 (1/5)

  Fold 1/10 | Seed 42
    BA=0.97826 | iter=2433 | 2m 5s
    Per-class: Low=0.9951 Medium=0.9740 High=0.9657

  Fold 2/10 | Seed 42
    BA=0.97640 | iter=2947 | 2m 10s
    Per-class: Low=0.9955 Medium=0.9747 High=0.9591

  Fold 3/10 | Se